# 📐 UtilScan — Entrena tu detector de útiles escolares

Este notebook afina (*fine-tuning*) **YOLOv8s pre-entrenado en Open Images V7** con un dataset de útiles escolares y lo exporta a `.tflite` para la app Android.

1. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`
2. Consigue tu API key gratis en https://app.roboflow.com/settings/api
3. Ejecuta todas las celdas (~20-40 min)

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" roboflow
!nvidia-smi -L

## 1. Dataset

Por defecto usa *Stationary Items Detection* (lápiz, lapicero, borrador, regla, tajador, tijeras, compás, libro…). Puedes cambiarlo por cualquier dataset de [Roboflow Universe](https://universe.roboflow.com/search?q=stationery) o combinar varios.

In [ ]:
ROBOFLOW_API_KEY = ""  # @param {type:"string"}
WORKSPACE = "hunaynzm"  # @param {type:"string"}
PROJECT = "stationary-items-detection"  # @param {type:"string"}
VERSION = 1  # @param {type:"integer"}

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print(open(DATA_YAML).read())

## 2. Entrenamiento

In [ ]:
from ultralytics import YOLO

EPOCHS = 100  # @param {type:"integer"}
model = YOLO("yolov8s-oiv7.pt")  # ya conoce Pen, Ruler, Eraser, Scissors, Pencil sharpener...
model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=640, batch=16, patience=25,
            degrees=15, flipud=0.2, mixup=0.1, name="school_supplies", exist_ok=True)
metrics = model.val()
print(f"mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}")

## 3. Prueba rápida

In [ ]:
import glob
from IPython.display import Image, display
best = YOLO(f"{model.trainer.save_dir}/weights/best.pt")
sample = sorted(glob.glob(f"{dataset.location}/test/images/*"))[:3]
for r in best.predict(sample, conf=0.35, save=True):
    display(Image(filename=f"{r.save_dir}/{r.path.split('/')[-1]}", width=480))

## 4. Exportar a TFLite y descargar

In [ ]:
import shutil, pathlib
from google.colab import files

best.export(format="tflite", imgsz=640, half=True)
tflite = next(pathlib.Path(model.trainer.save_dir).rglob("*_float16.tflite"))
shutil.copy(tflite, "school_supplies.tflite")
pathlib.Path("school_supplies_labels.txt").write_text("
".join(best.names[i] for i in range(len(best.names))) + "
")
files.download("school_supplies.tflite")
files.download("school_supplies_labels.txt")

## 5. Usarlo en la app

Copia los dos archivos a `app/src/main/assets/models/` y recompila. UtilScan detecta el modelo automáticamente y lo usa con prioridad (también puedes alternar entre modelos en *Ajustes*).